In [2]:
import pandas as pd
import time
# Load the CSV file
df = pd.read_csv("IOT Data Simulation/smart_logistic_tracker_japan.csv")

# Display the first 5 rows
df.head()

,timestamp,carrier,tracking_number,package_id,origin,current_location,delivery_location,prefecture,latitude,longitude,...,waiting_time_minutes,perishable,temperature,humidity,rfid_tag,rfid_verified,tamper_alert,traffic_status,inventory_level,asset_utilization
0,2026-05-04 13:50:26.857905,Yamato Transport,942646961460,PKG7545,Tokyo,Naha Central Post Office,Tokyo,Kanagawa,35.993159,139.038781,...,45,No,10.7,40,RFID736892,False,No,Heavy,99,84.59
1,2026-05-03 23:36:26.858095,Japan Post,74355111775,PKG2659,Tokyo,Nagoya Central Post Office,Kyoto,Kanagawa,35.691292,139.130870,...,54,Yes,6.2,82,RFID156229,False,Yes,Detour,363,53.39
2,2026-05-04 08:32:26.858217,Japan Post,217497030475,PKG7965,Osaka,Nagoya Central Post Office,Osaka,Aichi,35.591109,139.784940,...,144,Yes,-3.1,86,RFID890703,True,Yes,Heavy,25,95.75
3,2026-05-04 02:35:26.858332,Japan Post,249781996688,PKG5296,Fukuoka,Sapporo Central Post Office,Sapporo,Osaka,35.570440,139.689163,...,173,Yes,6.3,87,RFID603182,True,Yes,Detour,145,63.84
4,2026-05-04 08:28:26.858444,Japan Post,718415724062,PKG9987,Fukuoka,Yokohama Sales Office,Sapporo,Hokkaido,35.679375,139.408071,...,82,No,0.7,60,RFID921432,False,Yes,Detour,34,56.28


In [3]:
from web3 import Web3

# Connect to local blockchain
ganache_url = "http://127.0.0.1:8545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Verify connection
if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [4]:
# Replace with actual contract address from Remix
contract_address = "0xF633071fB31C49Fd5C805Cfd16e889F5F2952a67"


# Paste the ABI from Remix
abi = [
  {
    "inputs": [],
    "stateMutability": "nonpayable",
    "type": "constructor"
  },
  {
    "anonymous": False,
    "inputs": [
      {
        "indexed": False,
        "internalType": "uint256",
        "name": "timestamp",
        "type": "uint256"
      },
      {
        "indexed": False,
        "internalType": "string",
        "name": "packageId",
        "type": "string"
      },
      {
        "indexed": False,
        "internalType": "string",
        "name": "currentLocation",
        "type": "string"
      },
      {
        "indexed": False,
        "internalType": "string",
        "name": "status",
        "type": "string"
      }
    ],
    "name": "StatusUpdated",
    "type": "event"
  },
  {
    "inputs": [
      {
        "internalType": "string",
        "name": "_packageId",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "_location",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "_status",
        "type": "string"
      }
    ],
    "name": "storeStatus",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "index",
        "type": "uint256"
      }
    ],
    "name": "getPackageUpdate",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "",
        "type": "string"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "getTotalRecords",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "name": "logisticsRecords",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "timestamp",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "packageId",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "currentLocation",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "status",
        "type": "string"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "MAX_ENTRIES",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "owner",
    "outputs": [
      {
        "internalType": "address",
        "name": "",
        "type": "address"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  }
]  # Replace with your contract ABI

# Load the smart contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Set the default sender address (first account from Ganache)
web3.eth.default_account = web3.eth.accounts[0]

print(f"✅ Connected to Smart Contract at {contract_address}")

✅ Connected to Smart Contract at 0xF633071fB31C49Fd5C805Cfd16e889F5F2952a67


In [5]:
def send_iot_data(package_id, location, status):
    """
    Sends logistics IoT data
    to the deployed smart contract
    """

    txn = contract.functions.storeStatus(
        package_id,
        location,
        status
    ).transact({
        'from': web3.eth.default_account,
        'gas': 3000000
    })

    # Wait for transaction confirmation
    receipt = web3.eth.wait_for_transaction_receipt(txn)

    print(
        f"✅ Data Stored | {package_id} | "
        f"Location: {location} | "
        f"Status: {status} | "
        f"Txn Hash: {receipt.transactionHash.hex()}"
    )

# Store 100 records
for index, row in df.head(100).iterrows():

    package_id = str(row["package_id"])
    location = str(row["current_location"])
    status = str(row["latest_status"])

    send_iot_data(package_id, location, status)

    # Delay between transactions
    time.sleep(1)

print("\n✅ Successfully stored 100 records on the blockchain!")

Web3RPCError: {'message': 'VM Exception while processing transaction: revert', 'stack': 'RuntimeError: VM Exception while processing transaction: revert\n    at EIP1559FeeMarketTransaction.fillFromResult (/Applications/Ganache.app/Contents/Resources/static/node/node_modules/ganache/dist/node/1.js:2:12745)\n    at Miner.<anonymous> (/Applications/Ganache.app/Contents/Resources/static/node/node_modules/ganache/dist/node/1.js:2:36703)\n    at async Miner.<anonymous> (/Applications/Ganache.app/Contents/Resources/static/node/node_modules/ganache/dist/node/1.js:2:35116)\n    at async Miner.mine (/Applications/Ganache.app/Contents/Resources/static/node/node_modules/ganache/dist/node/1.js:2:39680)\n    at async Blockchain.mine (/Applications/Ganache.app/Contents/Resources/static/node/node_modules/ganache/dist/node/1.js:2:60063)\n    at async Promise.all (index 0)\n    at async TransactionPool.emit (/Applications/Ganache.app/Contents/Resources/static/node/node_modules/ganache/node_modules/emitt

In [ ]:
current_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {current_records}")

In [ ]:
# Retrieve and display the first stored record
first_record = contract.functions.getPackageUpdate(0).call()

print("📦 First Stored Record")
print(f"Timestamp: {first_record[0]}")
print(f"Package ID: {first_record[1]}")
print(f"Location: {first_record[2]}")
print(f"Status: {first_record[3]}")